In [1]:
# mount google drive
from google.colab import drive 
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Constants block
DATASET_NAME = "HR_128_normalized"
ALIGNED_DATASET_PATH = f"/content/drive/MyDrive/Datasets_aligned/{DATASET_NAME}"
MORPHS_PATH = f"/content/drive/MyDrive/Morphs/{DATASET_NAME}"
MASKING_METHOD = "blur" # either "sharp" or "blur" or "none" for the control group
BAD_PROBES = () # here probes used for morphing would be added, as they shal not be used anywhere else to avoid data leakage
TARGET_FMR = 0.01 # target false match rate for evaluation. As per EU standards it can be either a 0.1% or 1% FMR, so either 0.001 or 0.01
masking_range = (5,9) # provide a range from 0 to 12 
blur_kernel_size = 5 # provide an odd integer for the kernel size

# if method is "sharp" then the following masking parameters should be provided 
if MASKING_METHOD == "sharp" and not masking_range:
    raise ValueError("If masking method is 'sharp', you must provide a masking range.")

# if method is blur then the following masking parameters should be provided
if MASKING_METHOD == "blur" and (not masking_range or blur_kernel_size is None):
    raise ValueError("If masking method is 'blur', you must provide a masking range and a blur kernel size.")

In [3]:
%%bash
# Clonning elasticface and getting the pre-trained model checkpoint
# this one is bash script with strict exit on error

set -e

# clonning elasticface repo if it's not cloned already
if [ ! -d "ElasticFace" ]; then
    git clone "https://github.com/fdbtrs/ElasticFace"
fi

# installing gdown to get stuff from google drive
pip install -q gdown
# downloading the pre-trained model and doing some cleanup cause unfrotunately google drive just works that way
mkdir -p checkpoints
gdown 'https://drive.google.com/drive/folders/1q3ws_BQLmgXyiy2msvHummXq4pRqc1rx' --folder -O checkpoints
ls
# Clean up and rename
rm -f checkpoints/training.log
mv checkpoints/*.pth checkpoints/ElasticFaceArc.pth 2>/dev/null || true

echo "Setup completed successfully"

Processing file 1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb 295672backbone.pth
Processing file 1jGm6rHh-jJ40c34u5eXBgAhR3u4KHblH training.log
checkpoints
drive
ElasticFace
sample_data
Setup completed successfully


Cloning into 'ElasticFace'...
Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb
From (redirected): https://drive.google.com/uc?id=1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb&confirm=t&uuid=0f71239e-0aec-4b81-b74d-a6f7ef2bf560
To: /content/checkpoints/295672backbone.pth
100%|██████████| 261M/261M [00:04<00:00, 64.5MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1jGm6rHh-jJ40c34u5eXBgAhR3u4KHblH
To: /content/checkpoints/training.log
100%|██████████| 788k/788k [00:00<00:00, 7.36MB/s]
Download completed


In [4]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from itertools import combinations

# -------------------------------------------------------------
# 1. Corrected Masking Function (Channel-wise DFT as in Section 3.2)
# -------------------------------------------------------------
def create_sharp_mask(image_path: str, masking_range: tuple[int, int], target_size=(112, 112)):
    # Load image and resize to model's expected 112x112 input
    img = cv2.imread(image_path)
    img = cv2.resize(img, target_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Ensure RGB order
    
    h, w, _ = img.shape
    crow, ccol = h // 2, w // 2
    
    # Compute L2 radial distance matrix from center
    y, x = np.ogrid[:h, :w]
    radius = np.sqrt((x - ccol)**2 + (y - crow)**2)
    
    # 2D Mask: 0 inside [r_min, r_max], 1 elsewhere
    mask_2d = np.ones((h, w), dtype=np.float32)
    mask_2d[(radius >= masking_range[0]) & (radius <= masking_range[1])] = 0.0
    
    # Channel-wise DFT (OpenCV expects 1-channel for 2D real DFT)
    channels = cv2.split(np.float32(img))
    masked_channels = []
    for ch in channels:
        dft = cv2.dft(ch, flags=cv2.DFT_COMPLEX_OUTPUT)
        dft_shift = np.fft.fftshift(dft)
        dft_masked = dft_shift * mask_2d[:, :, np.newaxis] # Zero target band
        dft_ishift = np.fft.ifftshift(dft_masked)
        rec_ch = cv2.idft(dft_ishift, flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT)
        masked_channels.append(rec_ch)
        
    reconstructed_img = cv2.merge(masked_channels)
    
    # Convert to PyTorch Tensor and normalize to [-1, 1] (ElasticFace standard)
    tensor_img = torch.from_numpy(reconstructed_img).permute(2, 0, 1).contiguous().float()
    tensor_img = (tensor_img / 255.0 - 0.5) / 0.5
    return tensor_img.unsqueeze(0) # Shape: (1, 3, 112, 112)

def preprocess_unmasked_image(image_path: str, target_size=(112, 112)):
    """Preprocess the unmasked trusted probe image identically."""
    img = cv2.imread(image_path)
    img = cv2.resize(img, target_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor_img = torch.from_numpy(img).permute(2, 0, 1).contiguous().float()
    tensor_img = (tensor_img / 255.0 - 0.5) / 0.5
    return tensor_img.unsqueeze(0) # Shape: (1, 3, 112, 112) ALREADY TENSOR

In [5]:
def create_blur_mask(
    image_path: str,
    masking_range: tuple[int, int],
    blur_kernel_size: int,
    target_size=(112, 112)
):
    """Mask a radial DFT band and blur the reconstructed image.

    The frequency-band removal matches create_sharp_mask. The resulting
    spatial image is then blurred with the requested Gaussian kernel before
    being converted to the normalized ElasticFace input tensor.
    """
    if len(masking_range) != 2 or masking_range[0] < 0 or masking_range[1] < masking_range[0]:
        raise ValueError("masking_range must be a non-negative (min_radius, max_radius) tuple.")

    if blur_kernel_size <= 0 or blur_kernel_size % 2 == 0:
        raise ValueError("blur_kernel_size must be a positive odd integer.")

    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    img = cv2.resize(img, target_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w, _ = img.shape
    crow, ccol = h // 2, w // 2

    # Build the same radial frequency mask used by create_sharp_mask.
    y, x = np.ogrid[:h, :w]
    radius = np.sqrt((x - ccol) ** 2 + (y - crow) ** 2)
    mask_2d = np.ones((h, w), dtype=np.float32)
    mask_2d[(radius >= masking_range[0]) & (radius <= masking_range[1])] = 0.0

    # Apply the frequency mask independently to each color channel.
    channels = cv2.split(np.float32(img))
    reconstructed_channels = []
    for channel in channels:
        dft = cv2.dft(channel, flags=cv2.DFT_COMPLEX_OUTPUT)
        dft_shift = np.fft.fftshift(dft)
        dft_masked = dft_shift * mask_2d[:, :, np.newaxis]
        dft_ishift = np.fft.ifftshift(dft_masked)
        reconstructed_channel = cv2.idft(
            dft_ishift,
            flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT
        )
        reconstructed_channels.append(reconstructed_channel)

    reconstructed_img = cv2.merge(reconstructed_channels)
    reconstructed_img = np.clip(reconstructed_img, 0, 255).astype(np.uint8)

    # GaussianBlur requires an odd positive kernel in both dimensions.
    blurred_img = cv2.GaussianBlur(
        reconstructed_img,
        (blur_kernel_size, blur_kernel_size),
        sigmaX=0
    )

    tensor_img = torch.from_numpy(blurred_img).permute(2, 0, 1).contiguous().float()
    tensor_img = (tensor_img / 255.0 - 0.5) / 0.5
    return tensor_img.unsqueeze(0)  # Shape: (1, 3, 112, 112)


In [6]:
# Loading the model 
import sys 
import os
import torch 
import re

# appending repo to hte path to be able to import code from it
sys.path.append("./ElasticFace")
print(os.listdir("."))
from backbones.iresnet import iresnet100 # imporing the iresnet 100 architecture from the repo


# selecting the device to run the model on
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# initializing the model and moving it to the selected device
model = iresnet100()
model.to(device)

# loading the checkpoint into the model architecture
checkpoint = torch.load("./checkpoints/ElasticFaceArc.pth", map_location=device)
model.load_state_dict(checkpoint)

# switching the model to evaluation mode
model.eval()

# testing if the model is working by passing a fake tensor through it
fake_input = torch.randn(1, 3, 112, 112).to(device)
with torch.no_grad():
    fake_output = model(fake_input)
print(f"Fake output shape: {fake_output.shape}, if that works, the model is loaded and working correctly.")

['.config', 'ElasticFace', 'drive', 'checkpoints', 'sample_data']
Using device: cuda
Fake output shape: torch.Size([1, 512]), if that works, the model is loaded and working correctly.


In [7]:
# get ALL images from the aligned dataset as a list of paths by walking its structure
image_files = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(ALIGNED_DATASET_PATH)
    for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])

print(image_files[:5])  # print the first 5 image paths to verify

# Returns a list of dicts, one per morph:
# [{"morph": morph_path, "verification1": image_path, "verification2": image_path}, ...]
# The exact source probe names are extracted from the morph filename and excluded,
# while the remaining same-identity images are used as verification probes.
def get_evaluation_probes(morphs_path: str = MORPHS_PATH) -> list[dict]:
    results = []
    global BAD_PROBES  # Ensure we modify the global BAD_PROBES variable

    for morph in sorted(os.listdir(morphs_path)):
        if not morph.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        # getting the full path to the morph and it's name without the extension
        morph_path = os.path.join(morphs_path, morph)
        morph_name = os.path.splitext(os.path.basename(morph))[0]

        # using regex, extracting the identities which were used to construct the morph
        identity_1_name = re.search(r"(?<=^morphed_).*(?=_and)", morph_name).group(0)
        identity_2_name = re.search(r"(?<=_and_).*(?=$)", morph_name).group(0)

        # adding those identities to the BAD_PROBES tuple to avoid data leakage
        BAD_PROBES += (identity_1_name, identity_2_name)

        # getting identity prefixes to find other probes of the same identity
        identity_1_prefix = identity_1_name.split("_")[0]
        identity_2_prefix = identity_2_name.split("_")[0]

        # getting the paths to the other probes of the identities, excluding the morph identity itself
        identity_1_probes = sorted(
            file for file in image_files
            if os.path.basename(file).startswith(f"{identity_1_prefix}_")
            and os.path.splitext(os.path.basename(file))[0] != identity_1_name
        )
        identity_2_probes = sorted(
            file for file in image_files
            if os.path.basename(file).startswith(f"{identity_2_prefix}_")
            and os.path.splitext(os.path.basename(file))[0] != identity_2_name
        )

        if not identity_1_probes or not identity_2_probes:
            continue

        # getting first probe of each identity to use for verification
        results.append({
            "morph": morph_path,
            "verification1": identity_1_probes[0],
            "verification2": identity_2_probes[0],
        })

        # saving the third probe (second in the other probes list) for calculating the FNMR later
        if len(identity_1_probes) > 1:
            results[-1]["fnmr_calculation"] = identity_1_probes[1]

    return results


['/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00002/00002_940128_fa.png', '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00002/00002_940128_fb.png', '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00002/00002_940928_fa.png', '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00005/00005_930831_fa.png', '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00005/00005_941121_fa.png']


In [8]:
# getting the evaluation probes for the morphs in the dataset
evaluation_probes = get_evaluation_probes(MORPHS_PATH)

In [ ]:
# funcitons for batch processing images for fmr calibration and fnmr calculation later 
def build_tensor(image_path: str, masked:bool):
    if not masked: 
        return preprocess_unmasked_image(image_path=image_path)


    match MASKING_METHOD:
        case "none":
            # Baseline: unmasked reference photo
            tensor1 = preprocess_unmasked_image(img1_path).to(device)

        case "sharp":
            # MR-FR: Mask reference photo[cite: 1]
            tensor1 = create_sharp_mask(img1_path, masking_range).to(device)

        case "blur":
            tensor1 = create_blur_mask(
                img1_path, masking_range, blur_kernel_size
            ).to(device)
                
        case _:
            raise ValueError(f"Unknown masking method: {MASKING_METHOD}")

# this function gets batch of images and computes embeddins for them, this allows to 
# utilize more of the gpu while waiting for CPU and other I/O
def compute_embeddings(paths, masked, batch_size = 32): 
    embeddings = []

    # grabbing images in batches of 32 starting from the first one and to the end
    for start in range(0, len(paths), batch_size): 
        batch_paths = paths[start:start+batch_size]
        batch = torch.cat([
            build_tensor(path, masked)
            for path in batch_paths
        ]).to(device) # this concat stacks 32 images into 1 big tensor and lodas into gpu
        # this is what speeds up the processing instead of loading the single image all the time

        with torch.inference_mode(): 
            output = model.batch(batch)
            output = F.normalize(output, p=2, dim=1) # L2 normalizing for consine similarity

        embeddings.append(output)

    return torch.cat(embeddings)

In [9]:
# perform calibration of the model for the specific FMR value (either FMR100 or FMR1000 standarsd) and save the calibration threshold as a variable
# to use as a threshold for the verification task 

# building sutable identity dict 
calibration_dict = {} 

for image_path in image_files:
    identity_name = image_path.split("/")[-2]  # assuming the identity is the parent folder name
    if identity_name in calibration_dict.keys():
        continue # if it's there, that means we already have a probe for that identity so we skip the iteration
    else: 
        # checking if the image is not used anywhere in the morph creations once to avoid data leakage 
        if identity_name in BAD_PROBES:
            continue # if it's there, that means it's a bad probe and we skip the iteration
        else:
            calibration_dict[identity_name] = image_path # adding the probe to the dict

print(f"Calibration dict: {calibration_dict}")  # print the calibration dict to verify

# Determine which masking method is going to be used 
CALIBRATION_THRESHOLD = None
model.eval()
identities = list(calibration_dict.keys()) # or paths = [calibration_dict[identity] for identity in identities]

print(f"Calibrating on {len(identities)} identities...")

# The first image is processed according to MASKING_METHOD.
masked_embeddings = compute_embeddings(identities, masked=True)

# The second image is always an unmasked live probe.
unmasked_embeddings = compute_embeddings(identities, masked=False)

# Every row/column pair gives one impostor score.
score_matrix = masked_embeddings @ unmasked_embeddings.T # with transpositioned so no matching when we get upper triangle

# Keep only different-identity comparisons.
upper_triangle = torch.triu(
    torch.ones_like(score_matrix, dtype=torch.bool),
    diagonal=1
)

imposter_scores = score_matrix[upper_triangle].detach().cpu().numpy()
imposter_scores = np.array(imposter_scores)

print(f"Total imposter comparisons computed: {len(imposter_scores)}")

percentile = (1.0 - TARGET_FMR) * 100.0
CALIBRATION_THRESHOLD = float(np.percentile(imposter_scores, percentile))
print(f"Calibration completed. Threshold for FMR={TARGET_FMR}: {CALIBRATION_THRESHOLD:.4f}")

Calibration dict: {'00002': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00002/00002_940128_fa.png', '00005': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00005/00005_930831_fa.png', '00013': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00013/00013_930831_fb.png', '00019': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00019/00019_940128_fa.png', '00020': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00020/00020_930831_fa.png', '00024': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00024/00024_930831_fa.png', '00028': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00028/00028_930831_fb.png', '00029': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00029/00029_940128_fa.png', '00038': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00038/00038_930831_fb.png', '00040': '/content/drive/MyDrive/Datasets_ali

In [ ]:
# compute FNMR for the dataset and the specific masking method



In [10]:
# Image tensor transformation pipeline to prepare the images for the model
from PIL import Image
from torchvision import transforms

transform_pipeline = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(), # going from [0, 255] to [0.0, 1.0]
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # normalizing to [-1, 1]
])

In [11]:
# 2 funcitons which are used to determine how similar are the morphs to original people 
# first one gets the embedding of the image 
# second calculates the cosine similarity between two embeddings
def get_embedding(image_path: str) -> torch.Tensor:
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform_pipeline(image).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = model(image_tensor)

    if isinstance(embedding, (tuple, list)):
        embedding = embedding[0]

    # keep it on gpu
    embedding = embedding.detach().float()

    if embedding.ndim > 1:
        embedding = embedding.reshape(embedding.shape[0], -1)
        if embedding.shape[0] == 1:
            embedding = embedding[0]

    return embedding


def cosine_similarity(a: torch.Tensor, b: torch.Tensor) -> float:
    a = a.view(-1)
    b = b.view(-1)
    denom = (torch.norm(a) * torch.norm(b)).clamp_min(1e-12)
    return float(torch.dot(a, b) / denom)

In [12]:
import pandas as pd
import numpy as np

# make sure the model is in eval mode
model.eval()

match MASKING_METHOD: 
    case "none":
        CSV_OUTPUT_PATH = f"/content/drive/MyDrive/Morph_Evaluation/{DATASET_NAME}_cosine_similarity_{MASKING_METHOD}.csv"
    case "sharp": 
        CSV_OUTPUT_PATH = f"/content/drive/MyDrive/Morph_Evaluation/{DATASET_NAME}_cosine_similarity_{MASKING_METHOD}_{masking_range[0]}_{masking_range[1]}.csv"
    case "blur": 
        CSV_OUTPUT_PATH = F"/content/drive/MyDrive/Morph_Evaluation/{DATASET_NAME}_cosine_similarity_{MASKING_METHOD}_{masking_range[0]}_{masking_range[1]}_{blur_kernel_size}.csv"


os.makedirs(os.path.dirname(CSV_OUTPUT_PATH), exist_ok=True)


rows = []
for item in evaluation_probes:
    morph_path = item["morph"]
    verification1_path = item["verification1"]
    verification2_path = item["verification2"]

    if morph_path is None or verification1_path is None or verification2_path is None:
        continue

    # siwtch on how are we going to process the morph based on the selected masking method 
    match MASKING_METHOD:
        case "none":
            morph_embedding = get_embedding(morph_path)
        case "sharp":
            morph_masked_tensor = create_sharp_mask(morph_path, masking_range).to(device)
            with torch.no_grad():
                morph_embedding = model(morph_masked_tensor)
        case "blur":
                morph_masked_tensor = create_blur_mask(
                    morph_path, masking_range, blur_kernel_size
                ).to(device)
                with torch.no_grad():
                    morph_embedding = model(morph_masked_tensor)
        case _:
            raise ValueError(f"Unknown masking method: {MASKING_METHOD}")
        
    verification1_embedding = get_embedding(verification1_path)
    verification2_embedding = get_embedding(verification2_path)

    sim1 = cosine_similarity(morph_embedding, verification1_embedding)
    sim2 = cosine_similarity(morph_embedding, verification2_embedding)

    fooled = False 

    if sim1 >= CALIBRATION_THRESHOLD and sim2 >= CALIBRATION_THRESHOLD:
        fooled = True

    rows.append({
        "morph": morph_path,
        "verification1": verification1_path,
        "verification2": verification2_path,
        "1_similarity": sim1,
        "2_similarity": sim2,
        "fooled": fooled
    })

results_df = pd.DataFrame(rows, columns=["morph", "verification1", "verification2", "1_similarity", "2_similarity", "fooled"])
results_df.to_csv(CSV_OUTPUT_PATH, index=False)

print(f"Saved {len(results_df)} rows to: {CSV_OUTPUT_PATH}")
print(results_df.head())

Saved 46 rows to: /content/drive/MyDrive/Morph_Evaluation/filtered_dataset_ferret_3_cosine_similarity_blur_5_9_5.csv
                                               morph  \
0  /content/drive/MyDrive/Morphs/filtered_dataset...   
1  /content/drive/MyDrive/Morphs/filtered_dataset...   
2  /content/drive/MyDrive/Morphs/filtered_dataset...   
3  /content/drive/MyDrive/Morphs/filtered_dataset...   
4  /content/drive/MyDrive/Morphs/filtered_dataset...   

                                       verification1  \
0  /content/drive/MyDrive/Datasets_aligned/filter...   
1  /content/drive/MyDrive/Datasets_aligned/filter...   
2  /content/drive/MyDrive/Datasets_aligned/filter...   
3  /content/drive/MyDrive/Datasets_aligned/filter...   
4  /content/drive/MyDrive/Datasets_aligned/filter...   

                                       verification2  1_similarity  \
0  /content/drive/MyDrive/Datasets_aligned/filter...      0.322353   
1  /content/drive/MyDrive/Datasets_aligned/filter...      0.116523   